## PACOTES 

In [1]:
import os
import time

import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy.stats import gaussian_kde, ks_2samp

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_recall_curve,
    auc,
    matthews_corrcoef,
    log_loss,
    confusion_matrix
)

from openTSNE import TSNE


## CONF GERAIS

In [2]:
# BASE
RANK = 1
TARGET_COL_D = "status_fraude"
THRESHOLD_D = 0.50
estado_randomico_D = 42
NOME_HTML_D = f"3d_rank_{RANK}_tsne.html"

# HIPERPARÂMETROS GMM
numero_de_componentes_D = 2
inicializacoes_gausianas_D = 3
tipo_matriz_covariancia_D = "full"
erro_numerico_D = 1e-6

# HIPERPARÂMETROS t-SNE
TSNE_PERPLEXITY_D = 30
TSNE_N_ITER_D = 1500
TSNE_EARLY_EXAGGERATION_ITER_D = 100
TSNE_EARLY_EXAGGERATION_D = 12
TSNE_EXAGGERATION_D = 1
TSNE_LEARNING_RATE_D = "auto"
TSNE_METRIC_D = "euclidean"
TSNE_INITIALIZATION_D = "pca"
TSNE_NEGATIVE_GRADIENT_METHOD_D = "bh"
TSNE_N_JOBS_D = 7
TSNE_RANDOM_STATE_D = estado_randomico_D
TSNE_VERBOSE_D = True

## FUNCAO SCORE 

In [3]:
def calcular_score_final():
    auc_pr_norm = np.clip(auc_pr, 0, 1)
    mcc_norm = (mcc + 1) / 2
    mcc_norm = np.clip(mcc_norm, 0, 1)
    ks_norm = np.clip(ks, 0, 1)
    log_loss_norm = 1 / (1 + ll)
    score = (
        mcc_norm +
        ks_norm +
        log_loss_norm +
        auc_pr_norm
    ) / 4

    return round(float(score), 6)

## FUNCAO T-SNE/ORIGINAL 

In [4]:
def gerar_relatorio_1x1(
    RANK=1,
    usar_tsne=True,
    TARGET_COL="status_fraude",
    THRESHOLD=0.50,
    estado_randomico=42,

    numero_de_componentes=2,
    inicializacoes_gausianas=1,
    tipo_matriz_covariancia="full",
    erro_numerico=None,

    TSNE_PERPLEXITY=30,
    TSNE_N_ITER=1000,
    TSNE_EARLY_EXAGGERATION_ITER=100,
    TSNE_EARLY_EXAGGERATION=12,
    TSNE_EXAGGERATION=1,
    TSNE_LEARNING_RATE="auto",
    TSNE_METRIC="euclidean",
    TSNE_INITIALIZATION="pca",
    TSNE_NEGATIVE_GRADIENT_METHOD="bh",
    TSNE_N_JOBS=7,
    TSNE_VERBOSE=True
):

    def calcular_score_final():
        auc_pr_norm = np.clip(auc_pr, 0, 1)

        mcc_norm = (mcc + 1) / 2
        mcc_norm = np.clip(mcc_norm, 0, 1)

        ks_norm = np.clip(ks, 0, 1)

        log_loss_norm = 1 / (1 + ll)

        score = (
            auc_pr_norm +
            mcc_norm +
            ks_norm +
            log_loss_norm
        ) / 4

        return round(float(score), 6)

    tipo_relatorio = "tsne" if usar_tsne else "orig"
    NOME_HTML = f"1d_rank_{RANK}_{tipo_relatorio}.html"

    try:
        BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        BASE_DIR = os.getcwd()

    HTML_PATH = os.path.join(BASE_DIR, NOME_HTML)

    print("Diretório:", BASE_DIR)

    # LOAD RANKING 1x1
    df_scores = pd.read_csv("1x1_visu_scores.csv")

    row = df_scores[
        df_scores["Posicao_Rank"] == RANK
    ].iloc[0]

    feature_1 = row["Feature"]

    print(f"\nRank Selecionado: {RANK}")
    print(f"Feature: {feature_1}")

    # LOAD DATASET
    df = pd.read_csv("creditcard.csv")

    df_model = df[
        [feature_1, TARGET_COL]
    ].dropna().reset_index(drop=True)

    print("\nQuantidade usada:")
    print(df_model[TARGET_COL].value_counts())

    X_original = df_model[[feature_1]]
    y = df_model[TARGET_COL]

    # t-SNE
    if usar_tsne:

        print("\nRodando t-SNE 1D...\n")

        scaler_original = StandardScaler()
        X_scaled_original = scaler_original.fit_transform(X_original)

        inicio_tsne = time.perf_counter()

        tsne = TSNE(
            n_components=1,
            perplexity=TSNE_PERPLEXITY,
            learning_rate=TSNE_LEARNING_RATE,
            early_exaggeration_iter=TSNE_EARLY_EXAGGERATION_ITER,
            early_exaggeration=TSNE_EARLY_EXAGGERATION,
            n_iter=TSNE_N_ITER,
            exaggeration=TSNE_EXAGGERATION,
            metric=TSNE_METRIC,
            initialization=TSNE_INITIALIZATION,
            negative_gradient_method=TSNE_NEGATIVE_GRADIENT_METHOD,
            n_jobs=TSNE_N_JOBS,
            random_state=estado_randomico,
            verbose=TSNE_VERBOSE
        )

        X_tsne = tsne.fit(X_scaled_original)
        X_tsne = np.asarray(X_tsne)

        fim_tsne = time.perf_counter()

        print(
            f"\nt-SNE 1D finalizado em "
            f"{(fim_tsne - inicio_tsne):.2f} segundos."
        )

        df_model["TSNE_1"] = X_tsne[:, 0]

        cols_modelo = ["TSNE_1"]

        titulo_box = "Box Plot por Classe - t-SNE 1D"
        titulo_kde = "KDE - Distribuição Sobreposta após t-SNE 1D"
        titulo_prob = "Curva de Probabilidade do GMM - t-SNE 1D"
        titulo_relatorio = "Relatório GMM 1x1 após t-SNE 1D"

    else:

        cols_modelo = [feature_1]

        titulo_box = "Box Plot por Classe - Feature Original"
        titulo_kde = "KDE - Distribuição Sobreposta da Feature Original"
        titulo_prob = "Curva de Probabilidade do GMM - Feature Original"
        titulo_relatorio = "Relatório GMM 1x1 - Feature Original"

    # FEATURES PARA GMM
    X = df_model[cols_modelo]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # GMM
    gmm_params = {
        "n_components": numero_de_componentes,
        "covariance_type": tipo_matriz_covariancia,
        "random_state": estado_randomico,
        "n_init": inicializacoes_gausianas
    }

    if erro_numerico is not None:
        gmm_params["reg_covar"] = erro_numerico

    gmm = GaussianMixture(**gmm_params)

    gmm.fit(X_scaled)

    # CLUSTERS
    clusters = gmm.predict(X_scaled)

    ct = pd.crosstab(clusters, y)

    print("\nTabela Cluster x Classe Real:")
    print(ct)

    if 1 not in ct.columns:
        raise ValueError("Nenhuma fraude encontrada nos clusters.")

    cluster_fraude = ct[1].idxmax()

    print(f"\nCluster identificado como fraude: {cluster_fraude}")

    # PROBABILIDADES
    score = gmm.predict_proba(X_scaled)[:, cluster_fraude]

    score = np.clip(
        score,
        1e-15,
        1 - 1e-15
    )

    # PREDIÇÃO
    y_pred = (score >= THRESHOLD).astype(int)

    print("\nDistribuição score:")
    print(pd.Series(score).describe())

    print("\nQuantiles score:")
    print(np.quantile(score, [0, .1, .25, .5, .75, .9, .95, .99, 1]))

    print("\nPredições:")
    print(pd.Series(y_pred).value_counts())

    # MÉTRICAS
    prec, rec, _ = precision_recall_curve(y, score)

    auc_pr = auc(rec, prec)

    mcc = matthews_corrcoef(y, y_pred)

    ks = ks_2samp(
        score[y == 0],
        score[y == 1]
    ).statistic

    ll = log_loss(y, score)

    score_final = calcular_score_final()

    # MATRIZ CONFUSÃO
    cm = confusion_matrix(
        y,
        y_pred,
        labels=[0, 1]
    )

    cm_percent = (
        cm.astype(float)
        / cm.sum(axis=1)[:, np.newaxis]
    ) * 100

    texto_cm = []

    for i in range(2):

        linha = []

        for j in range(2):

            linha.append(
                f"{cm_percent[i, j]:.2f}%"
                f"<br>({cm[i, j]})"
            )

        texto_cm.append(linha)

    # DATASETS POR CLASSE
    df_nao_fraude = df_model[
        df_model[TARGET_COL] == 0
    ]

    df_fraude = df_model[
        df_model[TARGET_COL] == 1
    ]

    coluna_plot = cols_modelo[0]

    # FIGURA
    fig = make_subplots(
        rows=4,
        cols=2,

        specs=[
            [
                {"type": "box"},
                {"type": "heatmap"}
            ],
            [
                {"colspan": 2},
                None
            ],
            [
                {"colspan": 2},
                None
            ],
            [
                {"colspan": 2},
                None
            ]
        ],

        row_heights=[
            0.24,
            0.14,
            0.31,
            0.31
        ],

        horizontal_spacing=0.12,
        vertical_spacing=0.10,

        subplot_titles=(
            titulo_box,
            "Matriz de Confusão (%)",
            "",
            titulo_kde,
            titulo_prob
        )
    )

    # BOXPLOT NÃO FRAUDE
    fig.add_trace(
        go.Box(
            y=df_nao_fraude[coluna_plot],
            name="Não Fraude",
            boxpoints="outliers",
            marker=dict(
                color="rgba(0,0,255,0.45)"
            )
        ),
        row=1,
        col=1
    )

    # BOXPLOT FRAUDE
    fig.add_trace(
        go.Box(
            y=df_fraude[coluna_plot],
            name="Fraude",
            boxpoints="outliers",
            marker=dict(
                color="rgba(255,0,0,0.75)"
            )
        ),
        row=1,
        col=1
    )

    # HEATMAP CONFUSÃO
    fig.add_trace(
        go.Heatmap(
            z=cm_percent,
            x=[
                "Pred Não Fraude",
                "Pred Fraude"
            ],
            y=[
                "Real Não Fraude",
                "Real Fraude"
            ],
            text=texto_cm,
            texttemplate="%{text}",
            textfont=dict(size=18),
            colorscale="Blues",
            zmin=0,
            zmax=100,
            showscale=False
        ),
        row=1,
        col=2
    )

    # MÉTRICAS
    metricas = f"""
<b>MÉTRICAS</b><br><br>
AUC-PR: {auc_pr:.4f}<br>
MCC: {mcc:.4f}<br>
KS: {ks:.4f}<br>
Log Loss: {ll:.4f}<br>
Score Final: {score_final:.4f}
"""

    fig.add_trace(
        go.Scatter(
            x=[0.5],
            y=[0.5],
            mode="text",
            text=[metricas],
            textfont=dict(size=20),
            showlegend=False
        ),
        row=2,
        col=1
    )

    fig.update_xaxes(
        visible=False,
        range=[0, 1],
        row=2,
        col=1
    )

    fig.update_yaxes(
        visible=False,
        range=[0, 1],
        row=2,
        col=1
    )

    # KDE
    x_nao = df_nao_fraude[coluna_plot].dropna().values
    x_fraude = df_fraude[coluna_plot].dropna().values

    x_min = min(x_nao.min(), x_fraude.min())
    x_max = max(x_nao.max(), x_fraude.max())

    x_pad = (x_max - x_min) * 0.05

    x_grid = np.linspace(
        x_min - x_pad,
        x_max + x_pad,
        800
    )

    kde_nao = gaussian_kde(x_nao)
    kde_fraude = gaussian_kde(x_fraude)

    fig.add_trace(
        go.Scatter(
            x=x_grid,
            y=kde_nao(x_grid),
            mode="lines",
            name="KDE Não Fraude",
            line=dict(
                color="rgba(0,0,255,0.85)",
                width=3
            ),
            fill="tozeroy",
            opacity=0.45,
            showlegend=False
        ),
        row=3,
        col=1
    )

    fig.add_trace(
        go.Scatter(
            x=x_grid,
            y=kde_fraude(x_grid),
            mode="lines",
            name="KDE Fraude",
            line=dict(
                color="rgba(255,0,0,0.85)",
                width=3
            ),
            fill="tozeroy",
            opacity=0.45,
            showlegend=False
        ),
        row=3,
        col=1
    )

    # CURVA DE PROBABILIDADE DO GMM
    x_prob = np.linspace(
        df_model[coluna_plot].min(),
        df_model[coluna_plot].max(),
        1000
    ).reshape(-1, 1)

    x_prob_scaled = scaler.transform(x_prob)

    prob_gmm = gmm.predict_proba(
        x_prob_scaled
    )[:, cluster_fraude]

    fig.add_trace(
        go.Scatter(
            x=x_prob.flatten(),
            y=prob_gmm,
            mode="lines",
            name="Probabilidade GMM",
            line=dict(
                color="rgba(0,0,0,0.90)",
                width=4
            )
        ),
        row=4,
        col=1
    )

    fig.add_trace(
        go.Scatter(
            x=[
                df_model[coluna_plot].min(),
                df_model[coluna_plot].max()
            ],
            y=[
                THRESHOLD,
                THRESHOLD
            ],
            mode="lines",
            name=f"Threshold {THRESHOLD:.2f}",
            line=dict(
                color="rgba(255,0,0,0.80)",
                width=3,
                dash="dash"
            )
        ),
        row=4,
        col=1
    )

    # EIXOS
    fig.update_yaxes(
        title_text=coluna_plot,
        title_font=dict(size=20),
        tickfont=dict(size=14),
        row=1,
        col=1
    )

    fig.update_xaxes(
        title_text=coluna_plot,
        title_font=dict(size=22),
        tickfont=dict(size=15),
        row=3,
        col=1
    )

    fig.update_yaxes(
        title_text="Densidade",
        title_font=dict(size=22),
        tickfont=dict(size=15),
        row=3,
        col=1
    )

    fig.update_xaxes(
        title_text=coluna_plot,
        title_font=dict(size=22),
        tickfont=dict(size=15),
        row=4,
        col=1
    )

    fig.update_yaxes(
        title_text="Probabilidade de Fraude",
        title_font=dict(size=22),
        tickfont=dict(size=15),
        range=[0, 1],
        row=4,
        col=1
    )

    # DESCRIÇÃO
    if usar_tsne:
        descricao_features = f"""
        Feature original:
        {feature_1}
        <br>
        Nova feature:
        TSNE_1
        """
    else:
        descricao_features = f"""
        Feature original:
        {feature_1}
        """

    fig.update_layout(
        title=dict(
            text=f"""
            {titulo_relatorio}
            <br>
            Rank {RANK}
            <br>
            {descricao_features}
            <br>
            Base completa: 100% fraudes + 100% não fraudes
            <br>
            Corte: Probabilidade Cluster Fraude ≥ {THRESHOLD:.2f}
            """,

            x=0.5,
            y=0.985,

            xanchor="center",
            yanchor="top",

            font=dict(size=26)
        ),

        width=1900,
        height=2600,

        template="plotly_white",

        font=dict(size=18),

        margin=dict(
            t=360,
            b=160,
            l=120,
            r=120
        ),

        legend=dict(
            orientation="h",
            font=dict(size=18),
            yanchor="bottom",
            y=0.01,
            xanchor="center",
            x=0.5
        )
    )

    # SAVE HTML
    fig.write_html(
        HTML_PATH,
        include_plotlyjs="cdn"
    )

    print("\nHTML GERADO COM SUCESSO:")
    print(HTML_PATH)

    return {
        "HTML_PATH": HTML_PATH,
        "Rank": RANK,
        "Tipo": tipo_relatorio,
        "Feature": feature_1,
        "Feature_Modelo": coluna_plot,
        "AUC_PR": auc_pr,
        "MCC": mcc,
        "KS": ks,
        "Log_Loss": ll,
        "Score_Final": score_final
    }

## 1ST COLOCADO

### ORIGINAL 

In [5]:
resultado_1d_orig = gerar_relatorio_1x1(
    RANK=1,
    usar_tsne=False,
    TARGET_COL=TARGET_COL_D,
    THRESHOLD=THRESHOLD_D,
    estado_randomico=estado_randomico_D,

    numero_de_componentes=numero_de_componentes_D,
    inicializacoes_gausianas= inicializacoes_gausianas_D  ,
    tipo_matriz_covariancia=tipo_matriz_covariancia_D ,
    erro_numerico=erro_numerico_D,

    TSNE_PERPLEXITY=TSNE_PERPLEXITY_D,
    TSNE_N_ITER=TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER=TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION=TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION=TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC=TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado_1d_orig)


Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 1
Feature: V17

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Tabela Cluster x Classe Real:
status_fraude       0    1
row_0                     
0              277707  100
1                6608  392

Cluster identificado como fraude: 1

Distribuição score:
count    284807.000000
mean          0.050324
std           0.138224
min           0.013338
25%           0.014091
50%           0.016879
75%           0.026062
max           1.000000
dtype: float64

Quantiles score:
[0.01333759 0.01345394 0.01409086 0.01687861 0.02606153 0.06035222
 0.14937698 0.97698837 1.        ]

Predições:
0    277807
1      7000
Name: count, dtype: int64


c:\Users\Vitor Craveiro\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



HTML GERADO COM SUCESSO:
c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\1d_rank_1_orig.html
{'HTML_PATH': 'c:\\Users\\Vitor Craveiro\\Desktop\\UNESP\\MATÉRIAS FACULDADES\\MATÉRIAS 12 SEMESTRES\\TRABALHO DE CONCLUSAO DE CURSO 2\\1d_rank_1_orig.html', 'Rank': 1, 'Tipo': 'orig', 'Feature': 'V17', 'Feature_Modelo': 'V17', 'AUC_PR': 0.5305667576627885, 'MCC': 0.20745580482586187, 'KS': np.float64(0.7977763913808527), 'Log_Loss': 0.14764971754868272, 'Score_Final': 0.700854}


### T-SNE

In [6]:
resultado_1d_tsne = gerar_relatorio_1x1(
    RANK=1,
    usar_tsne=True,
    TARGET_COL=TARGET_COL_D,
    THRESHOLD=THRESHOLD_D,
    estado_randomico=estado_randomico_D,

    numero_de_componentes=numero_de_componentes_D,
    inicializacoes_gausianas= inicializacoes_gausianas_D  ,
    tipo_matriz_covariancia=tipo_matriz_covariancia_D ,
    erro_numerico=erro_numerico_D,

    TSNE_PERPLEXITY=TSNE_PERPLEXITY_D,
    TSNE_N_ITER=TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER=TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION=TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION=TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC=TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado_1d_orig)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 1
Feature: V17

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Rodando t-SNE 1D...

--------------------------------------------------------------------------------
TSNE(early_exaggeration=12, early_exaggeration_iter=100, exaggeration=1,
     n_components=1, n_iter=1500, n_jobs=7, negative_gradient_method='bh',
     random_state=42, verbose=True)
--------------------------------------------------------------------------------
===> Finding 90 nearest neighbors using Annoy approximate search using euclidean distance...
   --> Time elapsed: 216.73 seconds
===> Calculating affinity matrix...
   --> Time elapsed: 82.23 seconds
===> Calculating PCA-based initialization...
   --> Time elapsed: 0.13 seconds
===> Running optimization with exaggeration=12.00, lr=23733.92 for 100 iterations...
Iteration   50, KL divergen

c:\Users\Vitor Craveiro\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



HTML GERADO COM SUCESSO:
c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\1d_rank_1_tsne.html
{'HTML_PATH': 'c:\\Users\\Vitor Craveiro\\Desktop\\UNESP\\MATÉRIAS FACULDADES\\MATÉRIAS 12 SEMESTRES\\TRABALHO DE CONCLUSAO DE CURSO 2\\1d_rank_1_orig.html', 'Rank': 1, 'Tipo': 'orig', 'Feature': 'V17', 'Feature_Modelo': 'V17', 'AUC_PR': 0.5305667576627885, 'MCC': 0.20745580482586187, 'KS': np.float64(0.7977763913808527), 'Log_Loss': 0.14764971754868272, 'Score_Final': 0.700854}


## 2ND COLOCADO

### ORIGINAL 

In [7]:
resultado_1d_orig = gerar_relatorio_1x1(
    RANK=2,
    usar_tsne=False,
    TARGET_COL=TARGET_COL_D,
    THRESHOLD=THRESHOLD_D,
    estado_randomico=estado_randomico_D,

    numero_de_componentes=numero_de_componentes_D,
    inicializacoes_gausianas= inicializacoes_gausianas_D  ,
    tipo_matriz_covariancia=tipo_matriz_covariancia_D ,
    erro_numerico=erro_numerico_D,

    TSNE_PERPLEXITY=TSNE_PERPLEXITY_D,
    TSNE_N_ITER=TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER=TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION=TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION=TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC=TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado_1d_orig)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 2
Feature: V14

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Tabela Cluster x Classe Real:
status_fraude       0    1
row_0                     
0              246823   49
1               37492  443

Cluster identificado como fraude: 1

Distribuição score:
count    284807.000000
mean          0.252007
std           0.237524
min           0.109922
25%           0.115855
50%           0.140512
75%           0.251432
max           1.000000
dtype: float64

Quantiles score:
[0.10992155 0.11082953 0.11585477 0.14051216 0.25143226 0.64502609
 0.91556139 0.99998366 1.        ]

Predições:
0    246872
1     37935
Name: count, dtype: int64


c:\Users\Vitor Craveiro\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



HTML GERADO COM SUCESSO:
c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\1d_rank_2_orig.html
{'HTML_PATH': 'c:\\Users\\Vitor Craveiro\\Desktop\\UNESP\\MATÉRIAS FACULDADES\\MATÉRIAS 12 SEMESTRES\\TRABALHO DE CONCLUSAO DE CURSO 2\\1d_rank_2_orig.html', 'Rank': 2, 'Tipo': 'orig', 'Feature': 'V14', 'Feature_Modelo': 'V14', 'AUC_PR': 0.5747383650883889, 'MCC': 0.0939273796790633, 'KS': np.float64(0.8335712250339534), 'Log_Loss': 0.6258992690197923, 'Score_Final': 0.642579}


### T-SNE

In [8]:
resultado_1d_tsne = gerar_relatorio_1x1(
    RANK=2,
    usar_tsne=True,
    TARGET_COL=TARGET_COL_D,
    THRESHOLD=THRESHOLD_D,
    estado_randomico=estado_randomico_D,

    numero_de_componentes=numero_de_componentes_D,
    inicializacoes_gausianas= inicializacoes_gausianas_D  ,
    tipo_matriz_covariancia=tipo_matriz_covariancia_D ,
    erro_numerico=erro_numerico_D,

    TSNE_PERPLEXITY=TSNE_PERPLEXITY_D,
    TSNE_N_ITER=TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER=TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION=TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION=TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC=TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado_1d_orig)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 2
Feature: V14

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Rodando t-SNE 1D...

--------------------------------------------------------------------------------
TSNE(early_exaggeration=12, early_exaggeration_iter=100, exaggeration=1,
     n_components=1, n_iter=1500, n_jobs=7, negative_gradient_method='bh',
     random_state=42, verbose=True)
--------------------------------------------------------------------------------
===> Finding 90 nearest neighbors using Annoy approximate search using euclidean distance...
   --> Time elapsed: 225.07 seconds
===> Calculating affinity matrix...
   --> Time elapsed: 75.87 seconds
===> Calculating PCA-based initialization...
   --> Time elapsed: 0.07 seconds
===> Running optimization with exaggeration=12.00, lr=23733.92 for 100 iterations...
Iteration   50, KL divergen

c:\Users\Vitor Craveiro\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



HTML GERADO COM SUCESSO:
c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\1d_rank_2_tsne.html
{'HTML_PATH': 'c:\\Users\\Vitor Craveiro\\Desktop\\UNESP\\MATÉRIAS FACULDADES\\MATÉRIAS 12 SEMESTRES\\TRABALHO DE CONCLUSAO DE CURSO 2\\1d_rank_2_orig.html', 'Rank': 2, 'Tipo': 'orig', 'Feature': 'V14', 'Feature_Modelo': 'V14', 'AUC_PR': 0.5747383650883889, 'MCC': 0.0939273796790633, 'KS': np.float64(0.8335712250339534), 'Log_Loss': 0.6258992690197923, 'Score_Final': 0.642579}


## 3RD COLOCADO

### ORIGINAL 

In [9]:
resultado_1d_orig = gerar_relatorio_1x1(
    RANK=3,
    usar_tsne=False,
    TARGET_COL=TARGET_COL_D,
    THRESHOLD=THRESHOLD_D,
    estado_randomico=estado_randomico_D,

    numero_de_componentes=numero_de_componentes_D,
    inicializacoes_gausianas= inicializacoes_gausianas_D  ,
    tipo_matriz_covariancia=tipo_matriz_covariancia_D ,
    erro_numerico=erro_numerico_D,

    TSNE_PERPLEXITY=TSNE_PERPLEXITY_D,
    TSNE_N_ITER=TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER=TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION=TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION=TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC=TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado_1d_orig)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 3
Feature: V12

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Tabela Cluster x Classe Real:
status_fraude       0    1
row_0                     
0              251870   77
1               32445  415

Cluster identificado como fraude: 1

Distribuição score:
count    284807.000000
mean          0.202698
std           0.258800
min           0.064992
25%           0.069428
50%           0.087868
75%           0.168198
max           1.000000
dtype: float64

Quantiles score:
[0.0649921  0.06566726 0.06942781 0.08786839 0.16819795 0.62034614
 0.98595349 0.99999127 1.        ]

Predições:
0    251947
1     32860
Name: count, dtype: int64


c:\Users\Vitor Craveiro\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



HTML GERADO COM SUCESSO:
c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\1d_rank_3_orig.html
{'HTML_PATH': 'c:\\Users\\Vitor Craveiro\\Desktop\\UNESP\\MATÉRIAS FACULDADES\\MATÉRIAS 12 SEMESTRES\\TRABALHO DE CONCLUSAO DE CURSO 2\\1d_rank_3_orig.html', 'Rank': 3, 'Tipo': 'orig', 'Feature': 'V12', 'Feature_Modelo': 'V12', 'AUC_PR': 0.6134493462516485, 'MCC': 0.09480858070121645, 'KS': np.float64(0.7834616906216897), 'Log_Loss': 0.6546892680644312, 'Score_Final': 0.637165}


### T-SNE

In [10]:
resultado_1d_tsne = gerar_relatorio_1x1(
    RANK=3,
    usar_tsne=True,
    TARGET_COL=TARGET_COL_D,
    THRESHOLD=THRESHOLD_D,
    estado_randomico=estado_randomico_D,

    numero_de_componentes=numero_de_componentes_D,
    inicializacoes_gausianas= inicializacoes_gausianas_D  ,
    tipo_matriz_covariancia=tipo_matriz_covariancia_D ,
    erro_numerico=erro_numerico_D,

    TSNE_PERPLEXITY=TSNE_PERPLEXITY_D,
    TSNE_N_ITER=TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER=TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION=TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION=TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC=TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado_1d_orig)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 3
Feature: V12

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Rodando t-SNE 1D...

--------------------------------------------------------------------------------
TSNE(early_exaggeration=12, early_exaggeration_iter=100, exaggeration=1,
     n_components=1, n_iter=1500, n_jobs=7, negative_gradient_method='bh',
     random_state=42, verbose=True)
--------------------------------------------------------------------------------
===> Finding 90 nearest neighbors using Annoy approximate search using euclidean distance...
   --> Time elapsed: 212.47 seconds
===> Calculating affinity matrix...
   --> Time elapsed: 71.94 seconds
===> Calculating PCA-based initialization...
   --> Time elapsed: 0.03 seconds
===> Running optimization with exaggeration=12.00, lr=23733.92 for 100 iterations...
Iteration   50, KL divergen

c:\Users\Vitor Craveiro\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



HTML GERADO COM SUCESSO:
c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\1d_rank_3_tsne.html
{'HTML_PATH': 'c:\\Users\\Vitor Craveiro\\Desktop\\UNESP\\MATÉRIAS FACULDADES\\MATÉRIAS 12 SEMESTRES\\TRABALHO DE CONCLUSAO DE CURSO 2\\1d_rank_3_orig.html', 'Rank': 3, 'Tipo': 'orig', 'Feature': 'V12', 'Feature_Modelo': 'V12', 'AUC_PR': 0.6134493462516485, 'MCC': 0.09480858070121645, 'KS': np.float64(0.7834616906216897), 'Log_Loss': 0.6546892680644312, 'Score_Final': 0.637165}


## 4TH COLOCADO

### ORIGINAL 

In [11]:
resultado_1d_orig = gerar_relatorio_1x1(
    RANK=4,
    usar_tsne=False,
    TARGET_COL=TARGET_COL_D,
    THRESHOLD=THRESHOLD_D,
    estado_randomico=estado_randomico_D,

    numero_de_componentes=numero_de_componentes_D,
    inicializacoes_gausianas= inicializacoes_gausianas_D  ,
    tipo_matriz_covariancia=tipo_matriz_covariancia_D ,
    erro_numerico=erro_numerico_D,

    TSNE_PERPLEXITY=TSNE_PERPLEXITY_D,
    TSNE_N_ITER=TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER=TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION=TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION=TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC=TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado_1d_orig)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 4
Feature: V10

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Tabela Cluster x Classe Real:
status_fraude       0    1
row_0                     
0              278246  116
1                6069  376

Cluster identificado como fraude: 1

Distribuição score:
count    284807.000000
mean          0.046464
std           0.138244
min           0.011212
25%           0.011564
50%           0.013400
75%           0.022316
max           1.000000
dtype: float64

Quantiles score:
[0.01121183 0.01126721 0.01156437 0.01340022 0.02231632 0.06587572
 0.13076677 0.99911638 1.        ]

Predições:
0    278362
1      6445
Name: count, dtype: int64


c:\Users\Vitor Craveiro\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



HTML GERADO COM SUCESSO:
c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\1d_rank_4_orig.html
{'HTML_PATH': 'c:\\Users\\Vitor Craveiro\\Desktop\\UNESP\\MATÉRIAS FACULDADES\\MATÉRIAS 12 SEMESTRES\\TRABALHO DE CONCLUSAO DE CURSO 2\\1d_rank_4_orig.html', 'Rank': 4, 'Tipo': 'orig', 'Feature': 'V10', 'Feature_Modelo': 'V10', 'AUC_PR': 0.21484383198912466, 'MCC': 0.20743670467491626, 'KS': np.float64(0.7792441010335924), 'Log_Loss': 0.25764997056680417, 'Score_Final': 0.598235}


### T-SNE 

In [12]:
resultado_1d_tsne = gerar_relatorio_1x1(
    RANK=4,
    usar_tsne=True,
    TARGET_COL=TARGET_COL_D,
    THRESHOLD=THRESHOLD_D,
    estado_randomico=estado_randomico_D,

    numero_de_componentes=numero_de_componentes_D,
    inicializacoes_gausianas= inicializacoes_gausianas_D  ,
    tipo_matriz_covariancia=tipo_matriz_covariancia_D ,
    erro_numerico=erro_numerico_D,

    TSNE_PERPLEXITY=TSNE_PERPLEXITY_D,
    TSNE_N_ITER=TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER=TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION=TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION=TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC=TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado_1d_orig)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 4
Feature: V10

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Rodando t-SNE 1D...

--------------------------------------------------------------------------------
TSNE(early_exaggeration=12, early_exaggeration_iter=100, exaggeration=1,
     n_components=1, n_iter=1500, n_jobs=7, negative_gradient_method='bh',
     random_state=42, verbose=True)
--------------------------------------------------------------------------------
===> Finding 90 nearest neighbors using Annoy approximate search using euclidean distance...
   --> Time elapsed: 213.60 seconds
===> Calculating affinity matrix...
   --> Time elapsed: 75.17 seconds
===> Calculating PCA-based initialization...
   --> Time elapsed: 0.04 seconds
===> Running optimization with exaggeration=12.00, lr=23733.92 for 100 iterations...
Iteration   50, KL divergen

c:\Users\Vitor Craveiro\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


## 5TH COLOCADO

### ORIGINAL 

In [13]:
resultado_1d_orig = gerar_relatorio_1x1(
    RANK=5,
    usar_tsne=False,
    TARGET_COL=TARGET_COL_D,
    THRESHOLD=THRESHOLD_D,
    estado_randomico=estado_randomico_D,

    numero_de_componentes=numero_de_componentes_D,
    inicializacoes_gausianas= inicializacoes_gausianas_D  ,
    tipo_matriz_covariancia=tipo_matriz_covariancia_D ,
    erro_numerico=erro_numerico_D,

    TSNE_PERPLEXITY=TSNE_PERPLEXITY_D,
    TSNE_N_ITER=TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER=TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION=TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION=TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC=TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado_1d_orig)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 5
Feature: V16

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Tabela Cluster x Classe Real:
status_fraude       0    1
row_0                     
0              239965   87
1               44350  405

Cluster identificado como fraude: 1

Distribuição score:
count    284807.000000
mean          0.326175
std           0.209778
min           0.183996
25%           0.192141
50%           0.227567
75%           0.360635
max           1.000000
dtype: float64

Quantiles score:
[0.18399581 0.18527097 0.19214079 0.22756674 0.36063462 0.66296489
 0.87605009 0.99755977 1.        ]

Predições:
0    240052
1     44755
Name: count, dtype: int64


c:\Users\Vitor Craveiro\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



HTML GERADO COM SUCESSO:
c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\1d_rank_5_orig.html
{'HTML_PATH': 'c:\\Users\\Vitor Craveiro\\Desktop\\UNESP\\MATÉRIAS FACULDADES\\MATÉRIAS 12 SEMESTRES\\TRABALHO DE CONCLUSAO DE CURSO 2\\1d_rank_5_orig.html', 'Rank': 5, 'Tipo': 'orig', 'Feature': 'V16', 'Feature_Modelo': 'V16', 'AUC_PR': 0.4740012511359541, 'MCC': 0.07612953335793002, 'KS': np.float64(0.7007645676407523), 'Log_Loss': 0.5842716347103596, 'Score_Final': 0.586009}


### T-SNE 

In [14]:
resultado_1d_tsne = gerar_relatorio_1x1(
    RANK=5,
    usar_tsne=True,
    TARGET_COL=TARGET_COL_D,
    THRESHOLD=THRESHOLD_D,
    estado_randomico=estado_randomico_D,

    numero_de_componentes=numero_de_componentes_D,
    inicializacoes_gausianas= inicializacoes_gausianas_D  ,
    tipo_matriz_covariancia=tipo_matriz_covariancia_D ,
    erro_numerico=erro_numerico_D,

    TSNE_PERPLEXITY=TSNE_PERPLEXITY_D,
    TSNE_N_ITER=TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER=TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION=TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION=TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC=TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado_1d_orig)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 5
Feature: V16

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Rodando t-SNE 1D...

--------------------------------------------------------------------------------
TSNE(early_exaggeration=12, early_exaggeration_iter=100, exaggeration=1,
     n_components=1, n_iter=1500, n_jobs=7, negative_gradient_method='bh',
     random_state=42, verbose=True)
--------------------------------------------------------------------------------
===> Finding 90 nearest neighbors using Annoy approximate search using euclidean distance...
   --> Time elapsed: 199.44 seconds
===> Calculating affinity matrix...
   --> Time elapsed: 65.60 seconds
===> Calculating PCA-based initialization...
   --> Time elapsed: 0.16 seconds
===> Running optimization with exaggeration=12.00, lr=23733.92 for 100 iterations...
Iteration   50, KL divergen

c:\Users\Vitor Craveiro\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



HTML GERADO COM SUCESSO:
c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\1d_rank_5_tsne.html
{'HTML_PATH': 'c:\\Users\\Vitor Craveiro\\Desktop\\UNESP\\MATÉRIAS FACULDADES\\MATÉRIAS 12 SEMESTRES\\TRABALHO DE CONCLUSAO DE CURSO 2\\1d_rank_5_orig.html', 'Rank': 5, 'Tipo': 'orig', 'Feature': 'V16', 'Feature_Modelo': 'V16', 'AUC_PR': 0.4740012511359541, 'MCC': 0.07612953335793002, 'KS': np.float64(0.7007645676407523), 'Log_Loss': 0.5842716347103596, 'Score_Final': 0.586009}
